# 04_Explainability_and_Interventions.ipynb
# FAILSAFE - SHAP Explainability + Personalized Interventions

In [ ]:
!pip install shap -q

import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries Imported!')

In [ ]:
# Load model and data
model = joblib.load('model.pkl')
df = pd.read_csv('processed_student_data.csv')

FEATURE_NAMES = ['G1', 'G2', 'G3', 'absences', 'failures', 'studytime',
                  'Medu', 'Fedu', 'goout', 'Dalc', 'Walc', 'health']

X = df[FEATURE_NAMES]
y = df['at_risk']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Model and data loaded!')
print('Features:', FEATURE_NAMES)

In [ ]:
# SHAP Explainability using LinearExplainer
# LinearExplainer is efficient for Logistic Regression pipelines
X_train_scaled = model.named_steps['scaler'].transform(X_train)
X_test_scaled = model.named_steps['scaler'].transform(X_test)

explainer = shap.LinearExplainer(model.named_steps['lr'], X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)

print('SHAP values calculated!')
print('SHAP values shape:', shap_values.shape)

In [ ]:
# Global Feature Importance
plt.figure(figsize=(12, 6))
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES, 
                   plot_type='bar', max_display=12, show=False)
plt.title('Global Feature Importance (SHAP)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Beeswarm Plot
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES, max_display=12)

In [ ]:
# Personalized Intervention Generator
def generate_intervention(student_data, risk_score, shap_vals):
    risk_label = 'High' if risk_score >= 0.6 else ('Medium' if risk_score >= 0.35 else 'Low')
    prefix = '[Early Intervention] ' if risk_label == 'Medium' else ''
    
    print(f'Risk Score: {risk_score*100:.1f}% -> {risk_label} RISK')
    print()
    
    interventions = []
    
    if student_data.get('absences', 0) > 10:
        interventions.append(f"{prefix}High Absences ({int(student_data['absences'])}) -> Schedule immediate attendance counseling + parent meeting")
    
    if student_data.get('failures', 0) > 0:
        interventions.append(f"{prefix}Past Failures ({int(student_data['failures'])}) -> Enroll in remedial classes + assign academic mentor")
    
    if student_data.get('G3', 20) < 10:
        interventions.append(f"{prefix}Low Final Grade (G3={int(student_data['G3'])}/20) -> Personalized study plan + extra tutorial sessions")
    
    if student_data.get('studytime', 4) < 2:
        interventions.append(f"{prefix}Low Study Time -> Create structured weekly study schedule with faculty guidance")
    
    if student_data.get('goout', 1) > 3:
        interventions.append(f"{prefix}High Social Activity -> Refer to student counselor for time management guidance")
    
    if student_data.get('Dalc', 1) > 2 or student_data.get('Walc', 1) > 3:
        interventions.append(f"{prefix}Lifestyle Risk Factors -> Connect with student wellness counselor")
    
    if not interventions:
        interventions.append(f"{prefix}Multiple risk factors detected -> Assign academic mentor + bi-weekly review sessions")
    
    print('Personalized Intervention Plan:')
    for i, intervention in enumerate(interventions, 1):
        print(f'{i}. {intervention}')

# Test on a sample student
sample_idx = 0
student = X_test.iloc[sample_idx].to_dict()
risk_score = model.predict_proba(X_test.iloc[[sample_idx]])[0, 1]
shap_vals = shap_values[sample_idx]

generate_intervention(student, risk_score, shap_vals)

In [ ]:
# Generate interventions for all at-risk students
risk_scores = model.predict_proba(X_test)[:, 1]
at_risk_mask = risk_scores >= 0.35

print(f'Total students: {len(X_test)}')
print(f'High Risk (>=60%): {(risk_scores >= 0.6).sum()}')
print(f'Medium Risk (35-60%): {((risk_scores >= 0.35) & (risk_scores < 0.6)).sum()}')
print(f'Low Risk (<35%): {(risk_scores < 0.35).sum()}')
print(f'Students needing intervention: {at_risk_mask.sum()}')